# Verse Accentuation Diagrams

Hierarchical structure of Masoretic cantillation accents, following
**J.D. Price**, *The Syntax of Masoretic Accents in the Hebrew Bible*
(Temple Baptist Seminary, 2nd ed., 1990/2010).

Price's **Law of Continuous Dichotomy** (after Wickes): each disjunctive accent
governs a domain which splits recursively into a *near* segment (words immediately
preceding the governor) and a *remote* segment (farther words), each governed by
the next-strongest subordinate disjunctive.

**Hierarchy levels (prose books):**
| Level | Accent | Unicode |
|---|---|---|
| H1 | Soph Pasuq (SOP) | U+05C3 |
| H2 | Silluq (SIL), Athnach (ATH) | U+05BD†, U+0591 |
| H3 | Tiphcha (TIP), Little/Great Zaqeph (ZAQ/GZAQ), Segolta (SEG), Shalsheleth (SHAL) | U+0596, U+0594/5, U+0592, U+0593 |
| H4 | Tebir (TEB), Pashta/Yethib (PASH/YETH), Zarqa (ZAR)†, Rebia (REB) | U+059B, U+0599/A, U+05AE, U+0597 |
| H5 | Geresh (GER), Garshaim (GER2), Pazer/Great Telisha/Great Pazer (PAZ/GTEL/GPAZ), Legarmeh (LEG)‡ | U+059C/E, U+05A1/A0/9F |

† U+05BD is Meteg; it functions as Silluq on the last word before SOP.  
† U+05AE is named ZINOR in Unicode but is the disjunctive Zarqa (H4) in prose.  
‡ Legarmeh is detected as Munach (U+05A3) + Paseq (U+05C0) on the same word.

**Sections:**
1. Single-verse diagram
2. Verse tree inspection
3. Batch generation

In [ ]:
import sys
sys.path.insert(0, '../../..')

from IPython.display import Image, display
from src.bible_grammar.ot.cantillation import (
    parse_verse, render_verse,
    PROSE_BOOKS, POETRY_BOOKS,
    AccentNode, Word,
)

## 1. Single-verse diagram

`render_verse(book, chapter, verse)` produces a PNG and returns its path.

In [ ]:
out = render_verse('Gen', 1, 1, output_path='/tmp/gen_1_1.png')
display(Image(str(out)))

In [ ]:
# A longer verse with deeper nesting
out2 = render_verse('Gen', 1, 2, output_path='/tmp/gen_1_2.png')
display(Image(str(out2)))

## 2. Verse tree inspection

`parse_verse()` returns an `AccentNode` tree you can traverse programmatically.

In [ ]:
def print_tree(node: AccentNode, indent: int = 0) -> None:
    words_str = '  '.join(
        f"{w.text}[{w.accent_name}]" for w in node.words
    )
    print(' ' * indent + f'[{node.accent}]  {words_str}')
    for child in node.children:
        print_tree(child, indent + 4)

tree = parse_verse('Gen', 1, 1)
print('Genesis 1:1')
print_tree(tree)

In [ ]:
tree2 = parse_verse('Gen', 1, 2)
print('Genesis 1:2')
print_tree(tree2)

In [ ]:
# Count hierarchy depth per verse in a chapter
def tree_depth(node: AccentNode) -> int:
    if not node.children:
        return 0
    return 1 + max(tree_depth(c) for c in node.children)

print(f'{'Verse':>8}  {'Depth':>6}  {'Nodes':>6}')
for vs in range(1, 32):
    t = parse_verse('Gen', 1, vs)
    def _n(nd):
        return 1 + sum(_n(c) for c in nd.children)
    print(f'  Gen 1:{vs:<3}  {tree_depth(t):>6}  {_n(t):>6}')

## 3. Batch generation

Use the CLI build script for large batches:

In [ ]:
# Generate all diagrams for Genesis 1 (31 verses)
import subprocess
result = subprocess.run(
    ['python', 'scripts/build_cantillation_diagram.py', 'Gen', '1'],
    capture_output=True, text=True, cwd='../../..'
)
print(result.stdout[-1000:] if len(result.stdout) > 1000 else result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

In [ ]:
# Show the Gen 1:7 diagram (a verse with deeper nesting)
out7 = render_verse('Gen', 1, 7, output_path='/tmp/gen_1_7.png')
display(Image(str(out7)))